# GeoVision — Explore EuroSAT MS

Quick sanity check on the dataset: class counts, a sample tile rendered as RGB, and all 13 spectral bands visualized side by side.

In [ ]:
import sys, os, glob
from collections import Counter

REPO_DIR = '/content/drive/MyDrive/GeoVision/code'  # adjust if running locally
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import rasterio
import numpy as np
import matplotlib.pyplot as plt

from src.data_loading import collect_eurosat_files
from config import EUROSAT_CLASSES, SENTINEL2_BANDS

In [ ]:
candidates = glob.glob('/content/drive/MyDrive/GeoVision/data/raw/**/Forest', recursive=True)
DATA_ROOT = os.path.dirname(candidates[0])
print('Data root:', DATA_ROOT)

In [ ]:
files = collect_eurosat_files(DATA_ROOT)
counts = Counter(f.parent.name for f in files)
for cls in EUROSAT_CLASSES:
    print(f'{cls:25s} {counts[cls]:>5d}')
print(f'{"TOTAL":25s} {sum(counts.values()):>5d}')

In [ ]:
sample = files[len(files)//2]
with rasterio.open(sample) as src:
    img = src.read().astype(np.float32)  # [13, 64, 64]

# Sentinel-2 RGB = bands B04, B03, B02 (1-indexed) which are indices 3, 2, 1
rgb = np.stack([img[3], img[2], img[1]], axis=-1)
rgb = np.clip(rgb / np.percentile(rgb, 98), 0, 1)

plt.figure(figsize=(4, 4))
plt.imshow(rgb)
plt.title(f'{sample.parent.name} — RGB composite')
plt.axis('off')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    if i < 13:
        ax.imshow(img[i], cmap='gray')
        ax.set_title(SENTINEL2_BANDS[i])
    ax.axis('off')
fig.suptitle(f'{sample.parent.name} — all 13 bands', y=1.02)
plt.tight_layout()
plt.show()